### Chatbot And RAG Evaluation
#### Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

#### This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

#### How to create test datasets
#### How to run your RAG application on those datasets
#### How to measure your application's performance using different evaluation metrics
#### Overview
#### A typical RAG evaluation workflow consists of three main steps:

##### Creating a dataset with questions and their expected answers
Running your RAG application on those questions
Using evaluators to measure how well your application performed, looking at factors like:
Answer relevance
Answer accuracy
Retrieval quality
For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

In [11]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"

In [10]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {
                "answer": "A platform for observing and evaluating LLM applications"
            },
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
    ],
)

LangSmithConflictError: Conflict for /datasets. HTTPError('409 Client Error: Conflict for url: https://api.smith.langchain.com/datasets', '{"detail":"Dataset with this name already exists."}')

### define metrics (LLM as a Judge)


In [5]:
import openai
from langsmith import wrappers

openai_client = wrappers.wrap_openai(openai.OpenAI())

eval_instructions = (
    "You are an expert professor specialized in grading students' answers to questions."
)


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
    response = (
        openai_client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=[
                {"role": "system", "content": eval_instructions},
                {"role": "user", "content": user_content},
            ],
        )
        .choices[0]
        .message.content
    )

    return response == "CORRECT"

In [13]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.


def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

### Run Evalutaions

In [7]:
default_instructions = (
    "Respond to the users question in a short, concise manner (one short sentence)."
)


def my_app(
    question: str, model: str = "gpt-4o-mini", instructions: str = default_instructions
) -> str:
    return (
        openai_client.chat.completions.create(
            model=model,
            temperature=0,
            messages=[
                {"role": "system", "content": instructions},
                {"role": "user", "content": question},
            ],
        )
        .choices[0]
        .message.content
    )

In [8]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [14]:
## Run our evaluation
experiment_results = client.evaluate(
    ls_target,  ## Your AI system
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="openai-4o-mini-chatbot",
)

d:\langchain_practice_project\ragpractice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'openai-4o-mini-chatbot-6b83563e' at:
https://smith.langchain.com/o/50d3a742-e38f-4026-a1c7-1f8a3ba79b14/datasets/755ac17d-46c2-4461-8957-33c1bed9c77e/compare?selectedSessions=aee75a99-08b5-4556-acd1-9468d71415a5




5it [00:13,  2.72s/it]


In [15]:
def ls_target(input: str)->dict:
    return {"response":my_app(inputs["questions"], model="gpt-4-turbo")}

In [19]:
experiment_results=client.evaluate(
    ls_target, 
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="openai-4-turbo-chatbot"
)

View the evaluation results for experiment: 'openai-4-turbo-chatbot-103cc46f' at:
https://smith.langchain.com/o/50d3a742-e38f-4026-a1c7-1f8a3ba79b14/datasets/755ac17d-46c2-4461-8957-33c1bed9c77e/compare?selectedSessions=9d134459-1289-44fb-8369-1dca98ead950




0it [00:00, ?it/s]Error running target function: name 'inputs' is not defined
Traceback (most recent call last):
  File "d:\langchain_practice_project\ragpractice\.venv\Lib\site-packages\langsmith\evaluation\_runner.py", line 2062, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "d:\langchain_practice_project\ragpractice\.venv\Lib\site-packages\langsmith\run_helpers.py", line 849, in wrapper
    function_result = run_container["context"].run(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\MLInfo0025\AppData\Local\Temp\ipykernel_20364\1429316731.py", line 2, in ls_target
    return {"response":my_app(inputs["questions"], model="gpt-4-turbo")}
                              ^^^^^^
NameError: name 'inputs' is not defined. Did you mean: 'input'?
Error running evaluator <DynamicRunEvaluator correctness> on run 01a18b8a-a1ba-7781-8336-528988117360: KeyError('response')
Traceback (most recent call last):
  File "d:\langchain_practice_project\ragpractice\

## Evaluation For RAG

In [20]:
## RAG
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents from the URLs
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)

# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

# Add the document chunks to the "vector store" using OpenAIEmbeddings
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=OpenAIEmbeddings(),
)

# With langchain we can easily turn any vector store into a retrieval component:
retriever = vectorstore.as_retriever(k=6)

C:\Users\MLInfo0025\AppData\Local\Temp\ipykernel_20364\50547154.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


ModuleNotFoundError: No module named 'bs4'